# Data Pipeline Tutorial (`src/nano_llm/data.py`)

This notebook explains and demonstrates the data pipeline used for training.

You will learn:
- how Tiny Shakespeare is loaded and split
- how `chunk_text()` works
- how `ShakespeareDataset` builds `(x, y)` next-token pairs
- how `create_dataloaders()` returns train/val `DataLoader`s

In [1]:
from pathlib import Path
import sys

# Make src/ importable whether notebook cwd is repo root or docs/.
candidates = [Path.cwd(), Path.cwd().parent]
repo_root = next((p for p in candidates if (p / "src" / "nano_llm").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate repo root containing src/nano_llm")

sys.path.insert(0, str(repo_root / "src"))

from nano_llm.data import (
    load_tiny_shakespeare,
    chunk_text,
    ShakespeareDataset,
    create_dataloaders,
)
from nano_llm.tokenizer import CharTokenizer, BPETokenizer, ByteBPETokenizer


## 1) Load Tiny Shakespeare

`load_tiny_shakespeare(val_split=0.1)` downloads the dataset and returns `(train_text, val_text)`.

If network access is unavailable, it raises a runtime error.

In [2]:
train_text, val_text = load_tiny_shakespeare(val_split=0.1)
print("train chars:", len(train_text))
print("val chars  :", len(val_text))
print("train preview:")
print(train_text[:300])

train chars: 1003854
val chars  : 111540
train preview:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## 2) `chunk_text()` mechanics

`chunk_text(text, seq_len, stride)`:
- tokenizes with `CharTokenizer` internally
- yields fixed-length chunks of token ids
- defaults `stride=seq_len` (no overlap)

In [3]:
toy = "ROMEO: hello world\nJULIET: hi\n"
chunks = list(chunk_text(toy, seq_len=8, stride=4))

print("num chunks:", len(chunks))
print("first 3 chunks:")
for c in chunks[:3]:
    print(c)

num chunks: 6
first 3 chunks:
[9, 8, 7, 3, 8, 2, 1, 14]
[8, 2, 1, 14, 13, 16, 16, 17]
[13, 16, 16, 17, 1, 19, 17, 18]


## 3) `ShakespeareDataset`: next-token training pairs

Each item is:
- `x`: token ids from positions `[i, ..., i+seq_len-2]`
- `y`: token ids from positions `[i+1, ..., i+seq_len-1]`

So `y` is `x` shifted by one token.

In [4]:
tokenizer = CharTokenizer.from_text(train_text + val_text, add_special=False)
ds = ShakespeareDataset(train_text[:2000], tokenizer=tokenizer, seq_len=16, stride=8)

print("dataset length:", len(ds))
x, y = ds[0]
print("x shape:", x.shape, "y shape:", y.shape)
print("x ids:", x.tolist())
print("y ids:", y.tolist())
print("shift check (x[1:] == y[:-1]):", (x[1:] == y[:-1]).all().item())

dataset length: 248
x shape: torch.Size([15]) y shape: torch.Size([15])
x ids: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0]
y ids: [47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14]
shift check (x[1:] == y[:-1]): True


## 4) `create_dataloaders()` end-to-end

This function builds train and validation `DataLoader`s using `ShakespeareDataset`.

In [5]:
train_loader, val_loader = create_dataloaders(
    train_text=train_text,
    val_text=val_text,
    tokenizer=tokenizer,
    seq_len=64,
    batch_size=8,
)

xb, yb = next(iter(train_loader))
print("train batch x:", tuple(xb.shape), "train batch y:", tuple(yb.shape))
if val_loader is not None:
    xv, yv = next(iter(val_loader))
    print("val batch x  :", tuple(xv.shape), "val batch y  :", tuple(yv.shape))

train batch x: (8, 63) train batch y: (8, 63)
val batch x  : (8, 63) val batch y  : (8, 63)


## 5) Same dataset API with different tokenizers

`ShakespeareDataset` accepts:
- `CharTokenizer`
- `BPETokenizer`
- `ByteBPETokenizer`

Only token IDs change; the `(x, y)` shifting logic stays the same.

In [6]:
sample_text = train_text[:10000]

tok_char = CharTokenizer.from_text(sample_text, add_special=False)
tok_bpe = BPETokenizer.from_text(sample_text, vocab_size=256, word_boundary_aware=True)
tok_bpe_byte = ByteBPETokenizer.from_text(sample_text, vocab_size=256, word_boundary_aware=True)

for name, tok in [
    ("char", tok_char),
    ("bpe", tok_bpe),
    ("bpe_byte", tok_bpe_byte),
]:
    ds_tok = ShakespeareDataset(sample_text, tokenizer=tok, seq_len=64, stride=64)
    print(f"{name:8} vocab={tok.vocab_size:4d} dataset_len={len(ds_tok):4d}")

char     vocab=  57 dataset_len= 156
bpe      vocab= 256 dataset_len=  91
bpe_byte vocab= 256 dataset_len=  91


## 6) Common pitfalls and tips

- Very large `seq_len` can reduce dataset length a lot.
- Smaller `stride` increases overlap and dataset size.
- Always compare model quality with tokenizer-aware metrics (e.g., BPB) across tokenizer types.
- If dataset download fails in Docker, check container network access.